# RQ4: Decision Framework

**Research Question:** Which architecture should practitioners choose for their specific constraints?

## Overview

This notebook synthesizes findings from RQ1-RQ3 into a practical decision framework:
1. **Radar Chart**: Visualizes each architecture's strengths/weaknesses across 4 key dimensions
2. **Decision Tree**: Provides constraint-based selection guidance
3. **Hypothesis Summary**: Consolidates all H1-H3 outcomes

## Key Finding
There is no universally superior architecture. The optimal choice depends on specific constraints:
- **Budget-constrained**: Monolithic (2x cheaper than Triton)
- **Latency SLO**: Triton (best P99 stability at high load)
- **Throughput-focused**: Microservices (highest RPS)
- **Fast deployment**: Monolithic (10x faster than Triton)

In [ ]:
import sys
from pathlib import Path

# Get project root (works regardless of current working directory)
_notebook_dir = Path().resolve()
if _notebook_dir.name == 'notebooks' and _notebook_dir.parent.name == 'analysis':
    _project_root = _notebook_dir.parent.parent
else:
    _project_root = _notebook_dir

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

from analysis.utilities.loaders import ResultsLoader

# Load configuration
ResultsLoader._load_config_from_yaml()
ARCH_COLORS = ResultsLoader.ARCH_COLORS
ARCH_DISPLAY_NAMES = ResultsLoader.ARCH_DISPLAY_NAMES

# Publication-quality settings
plt.rcParams.update({
    'figure.figsize': (10, 8),
    'figure.dpi': 150,
    'font.size': 11,
    'font.family': 'sans-serif',
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
})

PLOTS_DIR = Path('../plots/rq4')
PLOTS_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
# Load experimental data
loader = ResultsLoader()
df = loader.load_summary()

# Print data overview
print("Data loaded successfully:")
print(f"  - Summary.csv: {len(df)} runs")
print(f"  - Architectures: {df['architecture'].unique().tolist()}")
print(f"  - User levels: {sorted(df['concurrent_users'].unique())}")

## 1. Architecture Comparison Radar Chart

Compares architectures across four key dimensions:
- **Latency Stability**: Lower P99-P50 gap at high load = better
- **Throughput**: Higher RPS at 100 users = better
- **Dev Velocity**: Faster deployment + lower LOC = better
- **Cost Efficiency**: Lower cost per request = better

Each axis normalized relative to best performer (best = 1.0).

In [ ]:
# Compute metrics at high load (100 users)
high_load = df[df['concurrent_users'] == 100]
metrics = high_load.groupby('architecture').agg({
    'client_p99_ms': 'mean',
    'client_p50_ms': 'mean',
    'throughput_rps': 'mean',
}).reset_index()

# Add latency stability (inverse of variance)
metrics['p99_p50_gap'] = metrics['client_p99_ms'] - metrics['client_p50_ms']

# Get container counts from config
container_counts = ResultsLoader.CONTAINER_COUNTS
metrics['containers'] = metrics['architecture'].map(container_counts)

# Cost index: containers / throughput (lower is better)
metrics['cost_index'] = metrics['containers'] / metrics['throughput_rps']

# Dev velocity proxy: use container count as simplicity metric
# (monolithic=1 container is simpler than distributed=2)
# Plus estimated deployment time ratios from cost_analysis_for_thesis.md:
# Monolithic ~57s, Microservices ~79s, Triton ~600s
deploy_time_estimate = {'monolithic': 57, 'microservices': 79, 'triton': 600}
metrics['deploy_time'] = metrics['architecture'].map(deploy_time_estimate)

# LOC estimates from cost_analysis_for_thesis.md:
# Monolithic ~374, Microservices ~752, Triton ~524
loc_estimate = {'monolithic': 374, 'microservices': 752, 'triton': 524}
metrics['total_loc'] = metrics['architecture'].map(loc_estimate)

# Normalize to 0-1 scale (relative to best)
# For "lower is better" metrics, invert so higher = better on radar
def normalize_lower_better(series):
    return series.min() / series  # Best (lowest) = 1.0

def normalize_higher_better(series):
    return series / series.max()  # Best (highest) = 1.0

radar_data = pd.DataFrame({
    'architecture': metrics['architecture'],
    'Latency Stability': normalize_lower_better(metrics['p99_p50_gap']),
    'Throughput': normalize_higher_better(metrics['throughput_rps']),
    'Dev Velocity': normalize_lower_better(metrics['deploy_time'] + metrics['total_loc']/10),  # Combine deployment + LOC
    'Cost Efficiency': normalize_lower_better(metrics['cost_index']),
})

print("Radar Chart Metrics (normalized, 1.0 = best):")
display(radar_data.round(3))

In [ ]:
# Radar chart implementation
categories = ['Latency Stability', 'Throughput', 'Dev Velocity', 'Cost Efficiency']
n_cats = len(categories)

# Compute angle for each category
angles = [n / float(n_cats) * 2 * np.pi for n in range(n_cats)]
angles += angles[:1]  # Complete the loop

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection='polar'))

for _, row in radar_data.iterrows():
    arch = row['architecture']
    values = [row[cat] for cat in categories]
    values += values[:1]  # Complete the loop

    ax.plot(angles, values, 'o-', linewidth=2, label=ARCH_DISPLAY_NAMES[arch],
            color=ARCH_COLORS[arch])
    ax.fill(angles, values, alpha=0.15, color=ARCH_COLORS[arch])

# Customize chart
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=11)
ax.set_ylim(0, 1.1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], size=9)
ax.legend(loc='upper right', bbox_to_anchor=(1.15, 1.1))
ax.set_title('Architecture Comparison: No Universal Winner', size=14, y=1.08)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'rq4_radar_chart.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nSaved: rq4_radar_chart.png")
print("\nKey insight: Each architecture excels in different dimensions.")

## 2. Decision Tree

A hybrid approach combining constraint-based filtering (first level) with threshold-based selection (second level).

### Level 1: Hard Constraints
- Budget constraint? -> Monolithic (2x cheaper than alternatives)
- Startup time constraint? -> Not Triton (10x slower deployment)
- Must use existing Triton infrastructure? -> Triton

### Level 2: Performance Thresholds
- Need P99 < 10s at 100 users? -> Triton or Monolithic
- Need throughput > 12 RPS? -> Microservices
- Need latency stability at scale? -> Triton

In [ ]:
# Create decision tree visualization
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.axis('off')

# Decision tree structure using boxes and arrows
def draw_box(ax, x, y, text, color='white', width=18, height=8):
    box = plt.Rectangle((x-width/2, y-height/2), width, height,
                         facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=9, wrap=True)

def draw_arrow(ax, x1, y1, x2, y2, label=''):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx, my+2, label, ha='center', va='bottom', fontsize=8, style='italic')

# Root node
draw_box(ax, 50, 90, 'Start:\nChoose ML Serving\nArchitecture', color='#f0f0f0')

# Level 1: Constraints
draw_box(ax, 25, 70, 'Budget\nConstrained?', color='#ffe6e6')
draw_arrow(ax, 50, 86, 25, 74)

draw_box(ax, 75, 70, 'Fast Startup\nRequired?', color='#ffe6e6')
draw_arrow(ax, 50, 86, 75, 74)

# Level 2: Threshold decisions
draw_box(ax, 15, 50, 'P99 SLO\n< 10s at 100u?', color='#e6f3ff')
draw_arrow(ax, 25, 66, 15, 54, 'No')

draw_box(ax, 35, 50, 'Throughput\n> 12 RPS?', color='#e6f3ff')
draw_arrow(ax, 25, 66, 35, 54, 'No')

draw_box(ax, 65, 50, 'Latency\nStability?', color='#e6f3ff')
draw_arrow(ax, 75, 66, 65, 54, 'No')

draw_box(ax, 85, 50, 'Existing\nTriton Infra?', color='#e6f3ff')
draw_arrow(ax, 75, 66, 85, 54, 'No')

# Recommendations (leaf nodes)
draw_box(ax, 10, 30, 'Monolithic', color=ARCH_COLORS['monolithic'], width=14, height=6)
draw_arrow(ax, 15, 46, 10, 33, 'Yes')
draw_arrow(ax, 25, 66, 10, 33, 'Yes')  # From budget constraint

draw_box(ax, 30, 30, 'Microservices', color=ARCH_COLORS['microservices'], width=14, height=6)
draw_arrow(ax, 35, 46, 30, 33, 'Yes')

draw_box(ax, 60, 30, 'Triton', color=ARCH_COLORS['triton'], width=14, height=6)
draw_arrow(ax, 65, 46, 60, 33, 'Yes')
draw_arrow(ax, 85, 46, 60, 33, 'Yes')

draw_box(ax, 90, 30, 'Not Triton\n(Mono/Micro)', color='#f0f0f0', width=14, height=6)
draw_arrow(ax, 75, 66, 90, 33, 'Yes')  # From fast startup

ax.set_title('Architecture Selection Decision Tree', fontsize=14, y=0.98)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'rq4_decision_tree.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: rq4_decision_tree.png")

## 3. Consolidated Hypothesis Outcomes

Summary of all hypothesis testing across RQ1 (Performance), RQ2 (Resource Efficiency), and RQ3 (Operational Complexity).

In [ ]:
# Try to load hypothesis results from each RQ
# If the files don't exist (notebooks haven't been run yet), create from known results

rq1_path = Path('../plots/rq1/rq1_hypothesis_results.csv')
rq2_path = Path('../plots/rq2/rq2_hypothesis_results.csv')
rq3_path = Path('../plots/rq3/rq3_hypothesis_results.csv')

all_results = []

# Try loading RQ1 results
if rq1_path.exists():
    rq1_results = pd.read_csv(rq1_path)
    rq1_results['RQ'] = 'RQ1: Performance'
    all_results.append(rq1_results)
    print(f"Loaded RQ1 results from {rq1_path}")
else:
    print(f"RQ1 results not found at {rq1_path} - using embedded data")
    # Embedded results from RQ1 notebook structure
    rq1_results = pd.DataFrame([
        {'Hypothesis': 'H1a', 'Statement': 'Monolithic lowest P99 at low load', 'Supported': True, 'Effect_Size': 'large', 'RQ': 'RQ1: Performance'},
        {'Hypothesis': 'H1b', 'Statement': 'Microservices overhead < 20%', 'Supported': True, 'Effect_Size': 'small', 'RQ': 'RQ1: Performance'},
        {'Hypothesis': 'H1c', 'Statement': 'Triton lower variance at high load', 'Supported': True, 'Effect_Size': 'large', 'RQ': 'RQ1: Performance'},
        {'Hypothesis': 'H1d', 'Statement': 'All saturate before 100 users', 'Supported': True, 'Effect_Size': 'N/A', 'RQ': 'RQ1: Performance'},
    ])
    all_results.append(rq1_results)

# Try loading RQ2 results
if rq2_path.exists():
    rq2_results = pd.read_csv(rq2_path)
    rq2_results['RQ'] = 'RQ2: Resource Efficiency'
    all_results.append(rq2_results)
    print(f"Loaded RQ2 results from {rq2_path}")
else:
    print(f"RQ2 results not found at {rq2_path} - using embedded data")
    # Embedded results from RQ2 notebook structure
    rq2_results = pd.DataFrame([
        {'Hypothesis': 'H2a', 'Statement': 'Monolithic lowest resource allocation', 'Supported': True, 'Effect_Size': 'N/A (design)', 'RQ': 'RQ2: Resource Efficiency'},
        {'Hypothesis': 'H2b', 'Statement': 'Microservices lower efficiency', 'Supported': True, 'Effect_Size': 'large', 'RQ': 'RQ2: Resource Efficiency'},
        {'Hypothesis': 'H2c', 'Statement': 'Triton higher baseline memory', 'Supported': True, 'Effect_Size': 'large', 'RQ': 'RQ2: Resource Efficiency'},
        {'Hypothesis': 'H2d', 'Statement': 'Efficiency converges at high load', 'Supported': True, 'Effect_Size': 'medium', 'RQ': 'RQ2: Resource Efficiency'},
    ])
    all_results.append(rq2_results)

# Try loading RQ3 results
if rq3_path.exists():
    rq3_results = pd.read_csv(rq3_path)
    rq3_results['RQ'] = 'RQ3: Operational Complexity'
    all_results.append(rq3_results)
    print(f"Loaded RQ3 results from {rq3_path}")
else:
    print(f"RQ3 results not found at {rq3_path} - using embedded data")
    # Embedded results from RQ3 notebook structure
    rq3_results = pd.DataFrame([
        {'Hypothesis': 'H3a', 'Statement': 'Triton requires fewest application LOC', 'Supported': False, 'Effect_Size': 'N/A', 'RQ': 'RQ3: Operational Complexity'},
        {'Hypothesis': 'H3b', 'Statement': 'Microservices requires most config LOC', 'Supported': True, 'Effect_Size': 'N/A', 'RQ': 'RQ3: Operational Complexity'},
        {'Hypothesis': 'H3c', 'Statement': 'Monolithic has shortest deployment time', 'Supported': True, 'Effect_Size': 'large', 'RQ': 'RQ3: Operational Complexity'},
    ])
    all_results.append(rq3_results)

# Consolidate
all_results_df = pd.concat(all_results, ignore_index=True)

# Reorder columns for display
display_cols = ['RQ', 'Hypothesis', 'Statement', 'Supported']
if 'Effect_Size' in all_results_df.columns:
    display_cols.append('Effect_Size')

# Select available columns
display_cols = [c for c in display_cols if c in all_results_df.columns]

print("\nConsolidated Hypothesis Outcomes (H1-H3)")
print("=" * 80)
display(all_results_df[display_cols])

# Summary statistics
supported_count = all_results_df['Supported'].apply(lambda x: x == True or str(x).lower() == 'true').sum()
total = len(all_results_df)
print(f"\nSupported: {supported_count}/{total} hypotheses")

In [ ]:
# Save consolidated results
all_results_df.to_csv(PLOTS_DIR / 'rq4_hypothesis_consolidated.csv', index=False)

# Save radar data as summary
radar_data.to_csv(PLOTS_DIR / 'rq4_summary.csv', index=False)

print("Saved outputs:")
print("  - rq4_hypothesis_consolidated.csv")
print("  - rq4_summary.csv")
print("  - rq4_radar_chart.png")
print("  - rq4_decision_tree.png")

## 4. Key Findings

### No Universal Winner
Each architecture excels in different scenarios:

| Scenario | Recommended | Rationale |
|----------|-------------|-----------|}
| Budget-constrained startup | Monolithic | 2x cheaper than Triton |
| High-throughput API | Microservices | Highest RPS (14.32 at 100 users) |
| Strict latency SLO | Triton | Lowest P99-P50 variance at scale |
| Rapid iteration | Monolithic | 10x faster deployment than Triton |
| Existing Triton infrastructure | Triton | Leverage existing investment |

### Practical Recommendations
1. **Start with Monolithic** for new projects (simplest, cheapest)
2. **Migrate to Microservices** when throughput becomes bottleneck
3. **Use Triton** when latency SLOs require predictable performance at scale

### Cost-Performance Trade-off
- Monolithic: Best cost efficiency (1.0 baseline)
- Microservices: 1.6x cost for 1.23x throughput
- Triton: 2x cost for similar throughput but better latency stability